In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier


In [2]:
train_raw = pd.read_csv("train.csv")
train_soft_label_raw = pd.read_csv("train_soft_label.csv")
test_raw = pd.read_csv("test.csv")
edges_raw = pd.read_csv("txs_edgelist.csv")

print("train_raw:", train_raw.shape)
print("train_soft_label_raw:", train_soft_label_raw.shape)
print("test_raw:", test_raw.shape)
print("edges_raw:", edges_raw.shape)


train_raw: (141772, 168)
train_soft_label_raw: (141772, 170)
test_raw: (15329, 168)
edges_raw: (234355, 2)


In [3]:
SOFT_LABEL_WEIGHT = 0.5
CONFIDENCE_CUTOFF = 0.7
RECENCY_DECAY = 0.03
USE_GRAPH_FEATURES = True
USE_TIME_STEP = False
SUBMISSION_PATH = "attempt_6_tuned_ensemble.csv"

ensemble_weights = {
    "LightGBM": 0.5,
    "XGBoost": 0.3,
    "ExtraTrees": 0.2,
}

lightgbm_params = {
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "num_leaves": 15,
    "n_estimators": 500,
    "min_split_gain": 0.1,
    "min_child_samples": 20,
    "max_depth": 10,
    "learning_rate": 0.05,
}

xgboost_params = {
    "subsample": 1.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "n_estimators": 300,
    "min_child_weight": 5,
    "max_depth": 7,
    "learning_rate": 0.03,
    "gamma": 0.0,
    "colsample_bytree": 0.8,
}

extra_trees_params = {
    "n_estimators": 500,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "min_impurity_decrease": 0.0,
    "max_features": 0.5,
    "max_depth": 40,
    "class_weight": "balanced",
}


In [4]:
node_times = pd.concat(
    [
        train_raw[["txId", "time_step"]],
        test_raw[["txId", "time_step"]],
    ],
    ignore_index=True,
).drop_duplicates("txId").set_index("txId")["time_step"]

edges = edges_raw[["txId1", "txId2"]].copy()
edges["time_1"] = edges["txId1"].map(node_times)
edges["time_2"] = edges["txId2"].map(node_times)

graph_feature_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
]

def temporal_node_features(cutoff_step):
    usable_edges = edges[
        edges["time_1"].notna()
        & edges["time_2"].notna()
        & (edges["time_1"] <= cutoff_step)
        & (edges["time_2"] <= cutoff_step)
    ]

    in_degree = usable_edges.groupby("txId2").size().rename("in_degree")
    out_degree = usable_edges.groupby("txId1").size().rename("out_degree")

    features = pd.concat([in_degree, out_degree], axis=1).fillna(0)
    features["total_degree"] = (
        features["in_degree"] + features["out_degree"]
    )
    features["degree_ratio"] = (
        features["out_degree"]
        / features["in_degree"].replace(0, np.nan)
    )
    features["degree_ratio"] = (
        features["degree_ratio"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return features.reset_index().rename(columns={"index": "txId"})

def add_temporal_graph_features(frame, cutoff_step):
    graph_features = temporal_node_features(cutoff_step)
    result = frame.merge(graph_features, on="txId", how="left")
    result[graph_feature_cols] = result[graph_feature_cols].fillna(0)
    return result

def calculate_recency_weights(time_steps, reference_time_step, decay_rate):
    age = reference_time_step - time_steps
    return np.exp(-decay_rate * age)


In [5]:
observed_mask = train_soft_label_raw["label"].notna()
soft_label_mask = (
    train_soft_label_raw["label"].isna()
    & train_soft_label_raw["p_illicit"].notna()
)
confidence_mask = (
    (train_soft_label_raw["p_illicit"] >= CONFIDENCE_CUTOFF)
    | (train_soft_label_raw["p_illicit"] <= 1 - CONFIDENCE_CUTOFF)
)
confident_pseudo_mask = soft_label_mask & confidence_mask

observed_rows = train_soft_label_raw.loc[observed_mask].copy()
observed_rows["training_target"] = observed_rows["label"].astype(int)
observed_rows["label_source_weight"] = 1.0

pseudo_rows = train_soft_label_raw.loc[confident_pseudo_mask].copy()
pseudo_rows["training_target"] = (
    pseudo_rows["p_illicit"] >= 0.5
).astype(int)
pseudo_rows["label_source_weight"] = SOFT_LABEL_WEIGHT

final_train = pd.concat(
    [observed_rows, pseudo_rows],
    ignore_index=True,
)

print("Confirmed rows:", len(observed_rows))
print("Confident pseudo-labelled rows:", len(pseudo_rows))
print("Total final training rows:", len(final_train))


Confirmed rows: 31235
Confident pseudo-labelled rows: 105509
Total final training rows: 136744


In [6]:
if USE_GRAPH_FEATURES:
    final_train = add_temporal_graph_features(
        final_train,
        cutoff_step=int(final_train["time_step"].max()),
    )

test_features = []
for step, step_rows in test_raw.groupby("time_step", sort=True):
    step_rows = step_rows.copy()
    if USE_GRAPH_FEATURES:
        step_rows = add_temporal_graph_features(
            step_rows,
            cutoff_step=int(step),
        )
    test_features.append(step_rows)

test_features = pd.concat(test_features, ignore_index=True)

feature_cols = [
    c for c in train_raw.columns
    if c not in ["index", "txId", "label", "time_step"]
]

final_feature_cols = list(feature_cols)
if USE_GRAPH_FEATURES:
    final_feature_cols += graph_feature_cols
if USE_TIME_STEP:
    final_feature_cols += ["time_step"]

X_final = final_train[final_feature_cols]
y_final = final_train["training_target"].astype(int)
X_test = test_features[final_feature_cols]

print("Feature count:", len(final_feature_cols))
print("Training shape:", X_final.shape)
print("Test shape:", X_test.shape)


Feature count: 169
Training shape: (136744, 169)
Test shape: (15329, 169)


In [7]:
class_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_final,
)

recency_weights = calculate_recency_weights(
    final_train["time_step"].to_numpy(),
    reference_time_step=int(final_train["time_step"].max()),
    decay_rate=RECENCY_DECAY,
)

source_weights = final_train["label_source_weight"].to_numpy()

final_sample_weights = (
    class_weights * recency_weights * source_weights
)
final_sample_weights = (
    final_sample_weights / final_sample_weights.mean()
)

print("Sample-weight minimum:", final_sample_weights.min())
print("Sample-weight maximum:", final_sample_weights.max())


Sample-weight minimum: 0.2389010638761486
Sample-weight maximum: 17.480467126166644


In [8]:
models = {
    "LightGBM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            objective="binary",
            random_state=100,
            n_jobs=-1,
            verbosity=-1,
            **lightgbm_params,
        )),
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=100,
            n_jobs=-1,
            **xgboost_params,
        )),
    ]),
    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            random_state=100,
            n_jobs=-1,
            **extra_trees_params,
        )),
    ]),
}

model_predictions = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(
        X_final,
        y_final,
        model__sample_weight=final_sample_weights,
    )
    model_predictions[model_name] = model.predict_proba(X_test)[:, 1]
    print(f"Finished {model_name}.")


Training LightGBM...
Finished LightGBM.
Training XGBoost...
Finished XGBoost.
Training ExtraTrees...
Finished ExtraTrees.


In [9]:
prediction_frame = pd.DataFrame(model_predictions)

for model_name in models:
    prediction_frame[f"{model_name}_rank"] = (
        prediction_frame[model_name].rank(pct=True)
    )

prediction_frame["ensemble_target"] = sum(
    ensemble_weights[model_name]
    * prediction_frame[f"{model_name}_rank"]
    for model_name in models
)

submission = pd.DataFrame({
    "index": test_raw["index"].to_numpy(),
    "target": prediction_frame["ensemble_target"].to_numpy(),
})

submission_path = SUBMISSION_PATH
submission.to_csv(submission_path, index=False)

display(submission.head())
print("Saved:", submission_path)
print("Prediction range:", submission["target"].min(), "to", submission["target"].max())
print("Unique target values:", submission["target"].nunique())


,index,target
0,0,0.498898
1,1,0.695851
2,2,0.672562
3,3,0.566527
4,4,0.175563


Saved: attempt_6_tuned_ensemble.csv
Prediction range: 0.028716811272750994 to 0.9996640354882902
Unique target values: 14824
